# Arctic Shadow Tracker - Main Workflow

**Simple pipeline: Sea cables + AIS-off vessels + Satellite imagery = Threat detection**

This notebook implements a straightforward workflow to detect vessels operating near submarine cables without AIS transponders - a key indicator of potentially suspicious maritime activity.

## Pipeline Overview
1. **Load cable locations** - Define protection zones around critical infrastructure
2. **Process satellite imagery** - Detect vessels in SAR images 
3. **Correlate with AIS data** - Find vessels without matching AIS signals
4. **Generate threat alerts** - Score and report anomalies

In [5]:
        "# Core imports and setup\n",
        "import pandas as pd\n",
        "import numpy as np\n",
        "import matplotlib.pyplot as plt\n",
        "import seaborn as sns\n",
        "from datetime import datetime, timedelta\n",
        "import sys\n",
        "import os\n",
        "\n",
        "# Add project modules to path - CRITICAL: Must point to project root\n",
        "import os\n",
        "project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))\n",
        "if project_root not in sys.path:\n",
        "    sys.path.insert(0, project_root)\n",
        "print(f\"📂 Project root: {project_root}\")\n",
        "\n",
        "# LEARNING PROGRESSION: Choose your complexity level\n",
        "# PHASE 1 (📚 Educational - for learning): Uncomment these imports\n",
        "# from detection.basic_vessel_detection import SimpleVesselDetector\n",
        "# from models.basic_autoencoder import SimpleAnomalyDetector\n",
        "\n",
        "# PHASE 2 (🔬 Research - for advanced analysis): These are the default imports\n",
        "try:\n",
        "    from detection.advanced_dark_vessels import DarkVesselDetector\n",
        "    from detection.advanced_cable_monitor import CableMonitor\n",
        "    from models.advanced_autoencoder import MaritimeAnomalyDetector\n",
        "    print(\"✅ Advanced modules loaded successfully\")\n",
        "except ImportError as e:\n",
        "    print(f\"❌ Import error: {e}\")\n",
        "    print(\"💡 Available files in detection/:\")\n",
        "    detection_files = os.listdir(os.path.join(project_root, 'detection'))\n",
        "    print(f\"   {detection_files}\")\n",
        "    print(\"💡 Available files in models/:\")\n",
        "    model_files = os.listdir(os.path.join(project_root, 'models'))\n",
        "    print(f\"   {model_files}\")\n",
        "    print(\"💡 See TROUBLESHOOTING.md for help\")\n",
        "    raise\n",
        "\n",
        "# Set up plotting style\n",
        "plt.style.use('default')\n",
        "sns.set_palette('husl')\n",
        "\n",
        "print(\"✅ Arctic Shadow Tracker initialized\")\n",
        "print(\"📚 Using ADVANCED modules - see PROGRESSION.md for educational alternatives\")\n",
        "print(\"📡 Loading maritime surveillance modules...\")"

'print("📡 Loading maritime surveillance modules...")'

In [6]:
# Initialize detection systems
print("🚢 Initializing vessel detection systems...")

# Initialize detectors with simple parameters
vessel_detector = DarkVesselDetector(
    matching_threshold_meters=1000,  # 1km for AIS-SAR correlation
    vessel_size_threshold=20,        # Minimum vessel size in pixels
    confidence_threshold=0.6         # Lower threshold for more detections
)

cable_monitor = CableMonitor(
    proximity_threshold_km=5,        # Alert within 5km of cables
    loitering_threshold_hours=1      # Alert after 1 hour loitering
)

anomaly_detector = MaritimeAnomalyDetector(
    input_dim=10,
    encoding_dim=5
)

print("✅ All detection systems ready")
print(f"📍 Monitoring {len(cable_monitor.cables)} submarine cables")
print(f"🎯 Cable protection zones: {cable_monitor.proximity_threshold}km radius")

🚢 Initializing vessel detection systems...


NameError: name 'DarkVesselDetector' is not defined

        "# Fetch REAL AIS and satellite data for Arctic region\n",
        "import requests\n",
        "import json\n",
        "\n",
        "def fetch_live_ais_data():\n",
        "    \"\"\"\n",
        "    Fetch live AIS data from publicly available APIs - REAL DATA ONLY\n",
        "    \"\"\"\n",
        "    print(\"🌐 Fetching LIVE AIS data from Arctic waters...\")\n",
        "    \n",
        "    # Arctic bounding box: Svalbard, Barents Sea, Norwegian Sea\n",
        "    bounds = {\n",
        "        'north': 82.0,\n",
        "        'south': 70.0, \n",
        "        'east': 35.0,\n",
        "        'west': 10.0\n",
        "    }\n",
        "    \n",
        "    ais_data = []\n",
        "    \n",
        "    # Try multiple real AIS data sources\n",
        "    apis_to_try = [\n",
        "        {\n",
        "            'name': 'AISHub Free API',\n",
        "            'url': f\"http://data.aishub.net/ws.php?username=DH_DEMO&format=1&output=json&compress=0&latmin={bounds['south']}&latmax={bounds['north']}&lonmin={bounds['west']}&lonmax={bounds['east']}\",\n",
        "            'timeout': 15\n",
        "        },\n",
        "        {\n",
        "            'name': 'MarineTraffic Public',\n",
        "            'url': 'https://services.marinetraffic.com/api/exportvessel/v:5',\n",
        "            'timeout': 10\n",
        "        },\n",
        "        {\n",
        "            'name': 'VesselFinder Public',\n",
        "            'url': 'https://www.vesselfinder.com/api/pub/vesselsearch',\n",
        "            'timeout': 10\n",
        "        }\n",
        "    ]\n",
        "    \n",
        "    for api in apis_to_try:\n",
        "        try:\n",
        "            print(f\"📡 Connecting to {api['name']}...\")\n",
        "            \n",
        "            if 'aishub' in api['url']:\n",
        "                response = requests.get(api['url'], timeout=api['timeout'])\n",
        "                if response.status_code == 200:\n",
        "                    data = response.json()\n",
        "                    if data.get('VESSELS'):\n",
        "                        for vessel in data['VESSELS'][:50]:  # Get more vessels\n",
        "                            ais_record = {\n",
        "                                'mmsi': str(vessel.get('MMSI', 'unknown')),\n",
        "                                'latitude': float(vessel.get('LATITUDE', 0)),\n",
        "                                'longitude': float(vessel.get('LONGITUDE', 0)),\n",
        "                                'speed': float(vessel.get('SOG', 0)),\n",
        "                                'course': float(vessel.get('COG', 0)),\n",
        "                                'timestamp': datetime.now().isoformat(),\n",
        "                                'vessel_name': vessel.get('SHIPNAME', f'VESSEL_{vessel.get(\"MMSI\", \"UNK\")}'),\n",
        "                                'vessel_type': vessel.get('SHIP_TYPE', 'unknown')\n",
        "                            }\n",
        "                            ais_data.append(ais_record)\n",
        "                        print(f\"✅ Retrieved {len(ais_data)} REAL AIS signals from {api['name']}\")\n",
        "                        break  # Success, stop trying other APIs\n",
        "                    else:\n",
        "                        print(f\"⚠️ {api['name']}: No vessels in Arctic region\")\n",
        "                else:\n",
        "                    print(f\"❌ {api['name']} error: HTTP {response.status_code}\")\n",
        "            else:\n",
        "                # For other APIs, would need API keys\n",
        "                print(f\"⚠️ {api['name']}: Requires API key - skipping\")\n",
        "                \n",
        "        except Exception as e:\n",
        "            print(f\"❌ {api['name']} failed: {e}\")\n",
        "            continue\n",
        "    \n",
        "    if not ais_data:\n",
        "        print(\"\")\n",
        "        print(\"🚨 NO REAL AIS DATA AVAILABLE\")\n",
        "        print(\"💡 Possible solutions:\")\n",
        "        print(\"   1. Check internet connection\")\n",
        "        print(\"   2. Try running at different time (APIs may be rate-limited)\")\n",
        "        print(\"   3. Sign up for AIS API keys (MarineTraffic, VesselFinder)\")\n",
        "        print(\"   4. Use local AIS data files if available\")\n",
        "        print(\"\")\n",
        "        \n",
        "        # Check for local real data files\n",
        "        import glob\n",
        "        local_ais_files = glob.glob('../data/ais/*.csv')\n",
        "        if local_ais_files:\n",
        "            print(f\"📂 Found {len(local_ais_files)} local AIS files - loading real data...\")\n",
        "            import pandas as pd\n",
        "            for file in local_ais_files[:1]:  # Load first file\n",
        "                try:\n",
        "                    df = pd.read_csv(file)\n",
        "                    for _, row in df.head(20).iterrows():\n",
        "                        ais_record = {\n",
        "                            'mmsi': str(row.get('mmsi', 'unknown')),\n",
        "                            'latitude': float(row.get('lat', 0)),\n",
        "                            'longitude': float(row.get('lon', 0)),\n",
        "                            'speed': float(row.get('speed', 0)),\n",
        "                            'course': float(row.get('course', 0)),\n",
        "                            'timestamp': row.get('timestamp', datetime.now().isoformat()),\n",
        "                            'vessel_name': row.get('vessel_name', f'VESSEL_{row.get(\"mmsi\", \"UNK\")}'),\n",
        "                            'vessel_type': row.get('vessel_type', 'unknown')\n",
        "                        }\n",
        "                        ais_data.append(ais_record)\n",
        "                    print(f\"✅ Loaded {len(ais_data)} REAL AIS records from local file\")\n",
        "                except Exception as e:\n",
        "                    print(f\"❌ Error loading {file}: {e}\")\n",
        "        \n",
        "        if not ais_data:\n",
        "            print(\"❌ NO REAL DATA AVAILABLE - Cannot proceed without real AIS data\")\n",
        "            print(\"🛑 Please ensure internet connection or provide local AIS data files\")\n",
        "            return []\n",
        "    \n",
        "    return ais_data\n",
        "\n",
        "def fetch_real_satellite_data():\n",
        "    \"\"\"\n",
        "    Fetch real satellite data or process local SAR imagery\n",
        "    \"\"\"\n",
        "    print(\"🛰️ Searching for REAL satellite data...\")\n",
        "    \n",
        "    # Check for local Sentinel-1 files\n",
        "    import glob\n",
        "    sentinel_files = glob.glob('../data/satellite/sentinel*.tif')\n",
        "    \n",
        "    if sentinel_files:\n",
        "        print(f\"📂 Found {len(sentinel_files)} local Sentinel-1 files\")\n",
        "        # Process real SAR imagery\n",
        "        sar_detections = []\n",
        "        for file in sentinel_files[:2]:  # Process first 2 files\n",
        "            print(f\"   🔍 Processing: {file}\")\n",
        "            try:\n",
        "                # Would use vessel_detector.detect_vessels_in_sar(file)\n",
        "                # For now, simulate processing of real file\n",
        "                detections = vessel_detector.detect_vessels_in_sar(file, roi_bounds=(70, 10, 82, 35))\n",
        "                sar_detections.extend(detections)\n",
        "                print(f\"   ✅ Detected {len(detections)} vessels in {file}\")\n",
        "            except Exception as e:\n",
        "                print(f\"   ⚠️ Could not process {file}: {e}\")\n",
        "        \n",
        "        if sar_detections:\n",
        "            print(f\"✅ Processed {len(sar_detections)} REAL SAR detections\")\n",
        "            return sar_detections\n",
        "    \n",
        "    # Try to download fresh Sentinel-1 data\n",
        "    try:\n",
        "        print(\"📡 Attempting to access Copernicus Open Access Hub...\")\n",
        "        # Would use sentinelsat API here\n",
        "        print(\"⚠️ Sentinel API requires credentials - configure for live data\")\n",
        "    except:\n",
        "        pass\n",
        "    \n",
        "    print(\"❌ NO REAL SATELLITE DATA AVAILABLE\")\n",
        "    print(\"💡 To get real satellite data:\")\n",
        "    print(\"   1. Download Sentinel-1 SAR images to ../data/satellite/\")\n",
        "    print(\"   2. Configure Copernicus API credentials\")\n",
        "    print(\"   3. Use ESA's Sentinel Hub or Google Earth Engine\")\n",
        "    \n",
        "    return []\n",
        "\n",
        "# Execute REAL data fetching only\n",
        "print(\"🔄 REAL DATA COLLECTION for Arctic Shadow Tracker\")\n",
        "print(\"=\" * 60)\n",
        "print(\"⚠️  WARNING: This system ONLY uses real data - no synthetic/demo data\")\n",
        "print(\"\")\n",
        "\n",
        "# Fetch all REAL data sources\n",
        "ais_data = fetch_live_ais_data()\n",
        "sar_detections = fetch_real_satellite_data()\n",
        "cable_locations = cable_monitor.cables  # Real cable database\n",
        "\n",
        "print(\"\\n📊 REAL DATA SUMMARY:\")\n",
        "print(f\"   🚢 REAL AIS signals: {len(ais_data)}\")\n",
        "print(f\"   🛰️ REAL SAR detections: {len(sar_detections)}\")\n",
        "print(f\"   🔌 Real cable locations: {len(cable_locations)}\")\n",
        "\n",
        "if len(ais_data) > 0:\n",
        "    print(\"\\n📍 Real vessels detected:\")\n",
        "    for vessel in ais_data[:5]:\n",
        "        print(f\"   📡 {vessel.get('vessel_name', 'Unknown')} (MMSI: {vessel['mmsi']}): {vessel['latitude']:.2f}°N, {vessel['longitude']:.2f}°E\")\n",
        "\n",
        "if len(sar_detections) > 0:\n",
        "    print(\"\\n🛰️ Real SAR detections:\")\n",
        "    for detection in sar_detections[:3]:\n",
        "        print(f\"   🎯 {detection['detection_id']}: {detection['latitude']:.2f}°N, {detection['longitude']:.2f}°E\")\n",
        "\n",
        "if len(ais_data) == 0:\n",
        "    print(\"\\n🛑 CANNOT PROCEED: No real AIS data available\")\n",
        "    print(\"Please check internet connection or provide local data files\")\n",
        "else:\n",
        "    print(\"\\n🎯 Ready for real threat detection analysis...\")"

In [7]:
# Fetch REAL AIS and satellite data for Arctic region
import requests
import json

def fetch_live_ais_data():
    """
    Fetch live AIS data from publicly available APIs
    """
    print("🌐 Fetching LIVE AIS data from Arctic waters...")
    
    # Arctic bounding box: Svalbard, Barents Sea, Norwegian Sea
    bounds = {
        'north': 82.0,
        'south': 70.0, 
        'east': 35.0,
        'west': 10.0
    }
    
    ais_data = []
    
    try:
        # Try AISHub free API
        print("📡 Connecting to AISHub API...")
        aishub_url = f"http://data.aishub.net/ws.php?username=DH_DEMO&format=1&output=json&compress=0&latmin={bounds['south']}&latmax={bounds['north']}&lonmin={bounds['west']}&lonmax={bounds['east']}"
        
        response = requests.get(aishub_url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get('VESSELS'):
                for vessel in data['VESSELS'][:20]:  # Limit to 20 vessels
                    ais_record = {
                        'mmsi': str(vessel.get('MMSI', 'unknown')),
                        'latitude': float(vessel.get('LATITUDE', 0)),
                        'longitude': float(vessel.get('LONGITUDE', 0)),
                        'speed': float(vessel.get('SOG', 0)),
                        'course': float(vessel.get('COG', 0)),
                        'timestamp': datetime.now().isoformat(),
                        'vessel_name': vessel.get('SHIPNAME', 'Unknown'),
                        'vessel_type': vessel.get('SHIP_TYPE', 'unknown')
                    }
                    ais_data.append(ais_record)
                print(f"✅ Retrieved {len(ais_data)} live AIS signals")
            else:
                print("⚠️ No vessels found in Arctic region")
        else:
            print(f"❌ AISHub API error: {response.status_code}")
            
    except Exception as e:
        print(f"❌ Error fetching live AIS: {e}")
    
    return ais_data

def fetch_sentinel_data():
    """
    Fetch Sentinel-1 SAR data for vessel detection
    """
    print("🛰️ Accessing Sentinel-1 SAR data...")
    
    try:
        # Check for local satellite data files
        import glob
        sentinel_files = glob.glob("../data/satellite/sentinel*.tif")
        
        detections = []
        if sentinel_files:
            print(f"📂 Found {len(sentinel_files)} local Sentinel files")
            # Process with vessel detection algorithm
            for file in sentinel_files:
                print(f"   Processing: {file}")
                # Would use vessel_detector.detect_vessels_in_sar(file)
        else:
            print("📁 No local Sentinel data found")
            print("💡 Configure Sentinel API or download sample data")
            
    except Exception as e:
        print(f"❌ Sentinel data error: {e}")
        
    return []

# Execute data fetching
print("🔄 Starting LIVE data collection for Arctic Shadow Tracker...")
print("=" * 60)

# Fetch all real data sources
live_ais_data = fetch_live_ais_data()
sentinel_detections = fetch_sentinel_data()  
cable_locations = cable_monitor.cables

print("\n📊 REAL DATA SUMMARY:")
print(f"   🚢 Live AIS signals: {len(live_ais_data)}")
print(f"   🛰️ Satellite detections: {len(sentinel_detections)}")
print(f"   🔌 Cable locations: {len(cable_locations)}")

if len(live_ais_data) > 0:
    print(f"\n✅ Successfully retrieved LIVE data")
    ais_data = live_ais_data
    print("📍 Sample vessels:")
    for vessel in live_ais_data[:3]:
        print(f"   {vessel.get('vessel_name', 'Unknown')}: {vessel['latitude']:.2f}°N, {vessel['longitude']:.2f}°E")
else:
    print("⚠️ No live data available - check API access and internet connection")
    ais_data = []

print("\n🎯 Ready for threat detection analysis...")

🔄 Starting LIVE data collection for Arctic Shadow Tracker...
🌐 Fetching LIVE AIS data from Arctic waters...
📡 Connecting to AISHub API...
❌ Error fetching live AIS: 'list' object has no attribute 'get'
🛰️ Accessing Sentinel-1 SAR data...
📁 No local Sentinel data found
💡 Configure Sentinel API or download sample data


NameError: name 'cable_monitor' is not defined

## Step 2: Core Pipeline - Detect Dark Vessels Near Cables

The heart of Arctic Shadow Tracker: correlate satellite detections with AIS data to find vessels operating without transponders near critical submarine cables.

In [8]:
# CORE PIPELINE: Process real data through threat detection
print("🎯 STARTING THREAT DETECTION PIPELINE")
print("=" * 50)

# Step 1: Process satellite detections (if available)
if len(sentinel_detections) > 0:
    print(f"🛰️ Processing {len(sentinel_detections)} satellite detections...")
    sar_detections = sentinel_detections
else:
    print("⚠️ No satellite detections available")
    print("💡 For production: Configure Sentinel-1 API or process SAR imagery")
    sar_detections = []

# Step 2: Find dark vessels (SAR detections without matching AIS)
print(f"\n🔍 Correlating satellite vs AIS data...")
print(f"   Satellite detections: {len(sar_detections)}")
print(f"   AIS signals: {len(ais_data)}")

if len(sar_detections) > 0 and len(ais_data) > 0:
    dark_vessels = vessel_detector.find_dark_vessels(
        sar_detections=sar_detections,
        ais_data=ais_data,
        time_tolerance_minutes=30
    )
    print(f"🚨 DARK VESSELS DETECTED: {len(dark_vessels)}")
else:
    print("⚠️ Cannot correlate - need both SAR and AIS data")
    dark_vessels = []

# Step 3: Check proximity to submarine cables  
print(f"\n🔌 Checking cable proximity for all vessels...")

# Combine all known vessels (AIS + dark vessels)
all_vessels = []

# Add AIS vessels
for ais_vessel in ais_data:
    vessel_info = {
        'vessel_id': ais_vessel['mmsi'],
        'latitude': ais_vessel['latitude'],
        'longitude': ais_vessel['longitude'],
        'timestamp': ais_vessel['timestamp'],
        'source': 'AIS',
        'vessel_name': ais_vessel.get('vessel_name', 'Unknown'),
        'has_ais': True
    }
    all_vessels.append(vessel_info)

# Add dark vessels  
for dark_vessel in dark_vessels:
    vessel_info = {
        'vessel_id': dark_vessel['detection_id'],
        'latitude': dark_vessel['latitude'],
        'longitude': dark_vessel['longitude'],
        'timestamp': dark_vessel['detection_time'],
        'source': 'SAR_DARK',
        'vessel_name': 'DARK_VESSEL',
        'has_ais': False,
        'risk_score': dark_vessel.get('risk_score', 0)
    }
    all_vessels.append(vessel_info)

# Check cable proximity for all vessels
vessels_near_cables = cable_monitor.check_vessel_cable_proximity(all_vessels)

# Step 4: Generate threat alerts
print(f"\n⚠️ THREAT ANALYSIS RESULTS:")
print("=" * 30)

high_threat_vessels = []
for vessel in vessels_near_cables:
    if vessel.get('near_cable', False):
        threat_level = 'LOW'
        if not vessel.get('has_ais', True):  # Dark vessel
            threat_level = 'HIGH'
        if vessel.get('distance_to_cable_km', 999) < 2:  # Very close
            threat_level = 'CRITICAL'
            
        high_threat_vessels.append({
            'vessel_id': vessel['vessel_id'],
            'threat_level': threat_level,
            'distance_to_cable': vessel.get('distance_to_cable_km', 'unknown'),
            'closest_cable': vessel.get('closest_cable', 'unknown'),
            'has_ais': vessel.get('has_ais', True),
            'latitude': vessel['latitude'],
            'longitude': vessel['longitude']
        })

# Display results
if len(high_threat_vessels) > 0:
    print(f"🚨 {len(high_threat_vessels)} VESSELS DETECTED NEAR CABLES:")
    for vessel in high_threat_vessels:
        ais_status = "✅ AIS ON" if vessel['has_ais'] else "❌ AIS OFF"
        print(f"   {vessel['threat_level']}: {vessel['vessel_id']} - {vessel['distance_to_cable']:.1f}km from {vessel['closest_cable']} - {ais_status}")
        
    # Count critical threats
    critical_threats = [v for v in high_threat_vessels if v['threat_level'] == 'CRITICAL']
    high_threats = [v for v in high_threat_vessels if v['threat_level'] == 'HIGH']
    
    print(f"\n📊 THREAT SUMMARY:")
    print(f"   🔴 CRITICAL: {len(critical_threats)} vessels")
    print(f"   🟡 HIGH: {len(high_threats)} vessels") 
    print(f"   🔵 TOTAL: {len(high_threat_vessels)} vessels near cables")
    
else:
    print("✅ No immediate threats detected near submarine cables")

print(f"\n🎯 Arctic Shadow Tracker analysis complete")

🎯 STARTING THREAT DETECTION PIPELINE
⚠️ No satellite detections available
💡 For production: Configure Sentinel-1 API or process SAR imagery

🔍 Correlating satellite vs AIS data...
   Satellite detections: 0


NameError: name 'ais_data' is not defined

## Step 3: Visualization and Reporting

Generate maps and threat reports for maritime security analysis.

In [9]:
# Generate comprehensive threat report and visualization
print("📋 GENERATING ARCTIC SHADOW TRACKER REPORT")
print("=" * 45)

# Create threat report
threat_report = {
    'report_timestamp': datetime.now().isoformat(),
    'analysis_region': 'Arctic Waters (70°N - 82°N)',
    'data_sources': {
        'ais_signals': len(ais_data),
        'satellite_detections': len(sar_detections), 
        'cables_monitored': len(cable_locations)
    },
    'threat_summary': {
        'total_vessels_analyzed': len(all_vessels) if 'all_vessels' in locals() else 0,
        'vessels_near_cables': len(high_threat_vessels) if 'high_threat_vessels' in locals() else 0,
        'critical_threats': len([v for v in high_threat_vessels if v['threat_level'] == 'CRITICAL']) if 'high_threat_vessels' in locals() else 0,
        'dark_vessels_detected': len(dark_vessels) if 'dark_vessels' in locals() else 0
    }
}

if 'high_threat_vessels' in locals() and len(high_threat_vessels) > 0:
    threat_report['threat_details'] = high_threat_vessels
    threat_report['recommendations'] = [
        "Increase surveillance of vessels without AIS near critical cables",
        "Deploy additional monitoring assets to high-threat areas", 
        "Coordinate with maritime authorities for vessel identification",
        "Continue real-time monitoring of cable protection zones"
    ]
else:
    threat_report['threat_details'] = []
    threat_report['recommendations'] = [
        "Continue routine monitoring of Arctic submarine cables",
        "Maintain vigilance for vessels operating without AIS transponders"
    ]

# Save report
import json
output_dir = "../outputs/daily_reports"
os.makedirs(output_dir, exist_ok=True)

report_filename = f"{output_dir}/arctic_threat_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(report_filename, 'w') as f:
    json.dump(threat_report, f, indent=2)

print(f"💾 Threat report saved: {report_filename}")

# Create visualization if we have vessel data
if 'all_vessels' in locals() and len(all_vessels) > 0:
    print("\n🗺️ Creating threat visualization map...")
    
    import folium
    
    # Create map centered on Arctic region
    arctic_map = folium.Map(
        location=[76.0, 20.0],  # Svalbard area
        zoom_start=4,
        tiles='OpenStreetMap'
    )
    
    # Add submarine cables
    for cable in cable_locations:
        cable_coords = [(lat, lon) for lon, lat in cable['route']]
        folium.PolyLine(
            cable_coords,
            weight=3,
            color='blue',
            opacity=0.8,
            popup=f"Cable: {cable['name']}"
        ).add_to(arctic_map)
        
        # Add cable protection zones
        for coord in cable['route'][::2]:  # Every other point to avoid clutter
            folium.Circle(
                location=[coord[1], coord[0]],  # lat, lon
                radius=5000,  # 5km protection zone
                color='lightblue',
                fill=True,
                fillOpacity=0.1,
                popup=f"Protection zone: {cable['name']}"
            ).add_to(arctic_map)
    
    # Add vessels
    for vessel in all_vessels:
        # Color based on threat level
        if vessel.get('has_ais', True):
            color = 'green'  # AIS vessel
            icon = 'ship'
        else:
            color = 'red'    # Dark vessel
            icon = 'warning-sign'
            
        # Check if near cable
        vessel_threats = [v for v in high_threat_vessels if v['vessel_id'] == vessel['vessel_id']] if 'high_threat_vessels' in locals() else []
        if vessel_threats:
            threat_level = vessel_threats[0]['threat_level']
            if threat_level == 'CRITICAL':
                color = 'darkred'
            elif threat_level == 'HIGH':
                color = 'orange'
        
        folium.Marker(
            location=[vessel['latitude'], vessel['longitude']],
            popup=f"Vessel: {vessel.get('vessel_name', vessel['vessel_id'])}<br>Source: {vessel['source']}<br>AIS: {'Yes' if vessel.get('has_ais', True) else 'No'}",
            icon=folium.Icon(color=color, icon=icon)
        ).add_to(arctic_map)
    
    # Save map
    map_filename = f"../outputs/visualizations/arctic_threat_map_{datetime.now().strftime('%Y%m%d_%H%M%S')}.html"
    os.makedirs("../outputs/visualizations", exist_ok=True)
    arctic_map.save(map_filename)
    print(f"🗺️ Threat map saved: {map_filename}")

# Display summary
print(f"\n📊 FINAL SUMMARY:")
print(f"   🌊 Analysis region: Arctic Waters")
print(f"   📡 AIS signals processed: {threat_report['data_sources']['ais_signals']}")
print(f"   🛰️ Satellite detections: {threat_report['data_sources']['satellite_detections']}")
print(f"   🔌 Cables monitored: {threat_report['data_sources']['cables_monitored']}")
print(f"   ⚠️ Vessels near cables: {threat_report['threat_summary']['vessels_near_cables']}")
print(f"   🚨 Critical threats: {threat_report['threat_summary']['critical_threats']}")
print(f"   👻 Dark vessels: {threat_report['threat_summary']['dark_vessels_detected']}")

print(f"\n✅ Arctic Shadow Tracker analysis complete!")
print(f"📋 Report: {report_filename}")
if 'map_filename' in locals():
    print(f"🗺️ Map: {map_filename}")

📋 GENERATING ARCTIC SHADOW TRACKER REPORT


NameError: name 'ais_data' is not defined